# Ajuste fino parcial de Video Swin-B: último stage + cabeza

Clasificación binaria de videos: **no explícito (0)** vs. **explícito (1)**, con los miniclips del preprocesamiento.

| Parte de Swin-B | Bloques | Estado |
|---|---|---|
| Patch embedding | — | congelado |
| Stage 1 (128 canales, 56×56) | 2 | congelado |
| Stage 2 (256 canales, 28×28) | 2 | congelado |
| Stage 3 (512 canales, 14×14) | 18 | congelado |
| **Stage 4** (1024 canales, 7×7) | **2** | **se entrena** |
| **Norm final + cabeza** (1024 → 1 con sigmoide, o 1024 → 2 con softmax) | — | **se entrena** |

Se entrenan unos 25 M de los 88 M de parámetros. Los stages congelados no guardan activaciones para el backward,
así que se usa mucha menos memoria que en el ajuste completo y no hace falta gradient checkpointing.

**Cabeza binaria (`BINARY_HEAD`)**
- `True`: una salida z, p(explícito) = σ(z), pérdida `BCEWithLogitsLoss`.
- `False`: dos salidas (z₀, z₁), p = softmax(z), pérdida `CrossEntropyLoss`.

Las dos opciones son equivalentes, porque softmax(z)₁ = σ(z₁ − z₀). La sigmoide solo quita un parámetro redundante.
Las métricas y la predicción por video (promedio de p de los K miniclips) se calculan igual con ambas.

**Partición fija (train / val / test).** Se crea una sola vez y se guarda en `SPLIT_DIR` como `train.txt`, `val.txt`
y `test.txt`. Todos los entrenamientos posteriores leen esos archivos, así que usan exactamente los mismos videos,
aunque cambie `SEED`.

Antes de empezar: **Entorno de ejecución → Cambiar tipo → GPU**. En Drive deben estar `Video-Swin-Transformer-Colab/`
y `salida/<conjunto>/` (con `miniclips/` y `meta/`).

In [ ]:
# Rutas en Drive (ajústalas)
PKG_DIR    = '/content/drive/MyDrive/Video-Swin-Transformer-Colab'
DATA_DRIVE = '/content/drive/MyDrive/salida/pornography2k'   # salida/<conjunto>
DATA_LOCAL = '/content/data/pornography2k'                   # copia local: lectura más rápida que desde Drive
SPLIT_DIR  = '/content/drive/MyDrive/vswin_splits/pornography2k'   # partición compartida por todos los entrenamientos
RUN_DIR    = '/content/drive/MyDrive/vswin_runs/p2k_swin_b_stage4_sigmoid_seed0'

# Partición: solo se usa la primera vez, cuando SPLIT_DIR aún no tiene los .txt
SPLIT_SEED    = 0
VAL_FRACTION  = 0.15
TEST_FRACTION = 0.15

# Entrenamiento: puede cambiar entre corridas sin afectar la partición
VARIANT       = 'swin_b_k400_22k'
FROZEN_STAGES = 3          # congela patch embedding + stages 1, 2 y 3
BINARY_HEAD   = True       # True: 1 salida + sigmoide (BCE) | False: 2 salidas + softmax (CE)
CLASS_NAMES   = ['no_explicito', 'explicito']
SEED          = 0          # inicialización de la cabeza, orden de muestreo y volteos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "$PKG_DIR"
!ffmpeg -hide_banner -version | head -n 1

In [ ]:
import json, torch, vswin
print(vswin.describe_environment())
device = vswin.default_device()

## 1. Datos y partición fija

Las etiquetas salen de la ruta original guardada en `meta['ruta']`. En Pornography-2k los archivos se llaman `vPorn…` y `vNonPorn…`.
Para otro conjunto (p. ej. LSPD), usa un CSV `video_id,label` o un dict `{video_id: clase}`.

`get_or_create_split`:
- **Primera vez:** divide por video y estratificado por clase (70/15/15 por defecto), y escribe en `SPLIT_DIR` los archivos
  `train.txt`, `val.txt`, `test.txt` y `split_info.json` (semilla, fracciones y conteos).
- **Corridas siguientes:** solo lee esos archivos. Si falta algún video de la lista (por ejemplo, uno que falló al
  preprocesar o que no se copió), se detiene con un error, porque la partición ya no sería la misma.

Para usar los folds oficiales, copia sus listas como `train.txt`, `val.txt` y `test.txt` en `SPLIT_DIR`.
Un archivo que falte se toma como lista vacía.

**Test** no se usa ni para elegir la época ni para ajustar hiperparámetros. Solo se evalúa al final (sección 5).

In [ ]:
!mkdir -p "$DATA_LOCAL" && rsync -a --info=progress2 "$DATA_DRIVE/" "$DATA_LOCAL/"

records = vswin.index_dataset(DATA_LOCAL, labels=vswin.label_from_path({'NonPorn': 0, 'Porn': 1}))
# records = vswin.index_dataset(DATA_LOCAL, labels='/content/drive/MyDrive/etiquetas.csv', class_names=CLASS_NAMES)

train_rec, val_rec, test_rec = vswin.get_or_create_split(
    records, SPLIT_DIR, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SPLIT_SEED)

print('train:', vswin.describe_records(train_rec, CLASS_NAMES))
print('val:  ', vswin.describe_records(val_rec, CLASS_NAMES))
print('test: ', vswin.describe_records(test_rec, CLASS_NAMES))
print(open(f'{SPLIT_DIR}/split_info.json').read())

In [ ]:
train_loader, val_loader = vswin.make_loaders(
    train_rec, val_rec,
    batch_size=4,           # T4: 4 miniclips por paso sin checkpointing (los stages 1–3 no guardan activaciones)
    num_workers=2,
    flip_prob=0.5,          # volteo horizontal, como el pipeline oficial
    balanced_sampling=True, # mismo peso por video (sin importar su K) y por clase
    seed=SEED,
)
test_loader = vswin.make_eval_loader(test_rec, num_workers=2)   # por video, sin aumentos

batch = next(iter(train_loader))
print(batch['clip'].shape, batch['clip'].dtype, batch['label'])   # (4, 32, 224, 224, 3) uint8

## 2. Modelo: backbone preentrenado con los stages 1–3 congelados

`frozen_stages=3` pone `requires_grad=False` en el patch embedding y en los stages 1–3, y los deja en modo `eval()`
(sin dropout ni drop path), incluso cuando se llama a `model.train()`.
La cabeza nueva (1 o 2 salidas, según `BINARY_HEAD`) empieza desde cero, y la de 400 clases de Kinetics no se carga.

In [ ]:
vswin.set_seed(SEED)
model = vswin.build_model(VARIANT, num_classes=2, binary=BINARY_HEAD, frozen_stages=FROZEN_STAGES,
                          use_checkpoint=False, device=device)
print('head:', model.cls_head.fc_cls, '| sigmoid' if model.binary else '| softmax')

total, trainable = vswin.count_parameters(model)
print(f'parameters: {total / 1e6:.1f} M total, {trainable / 1e6:.1f} M trainable ({100 * trainable / total:.0f} %)')

# Detalle por parte: confirma que solo el stage 4, la norm final y la cabeza tienen gradiente
parts = {'patch_embed': model.backbone.patch_embed,
         **{f'stage {i + 1} ({len(l.blocks)} blocks)': l for i, l in enumerate(model.backbone.layers)},
         'final norm': model.backbone.norm, 'head': model.cls_head}
for name, module in parts.items():
    n = sum(p.numel() for p in module.parameters())
    t = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f'{name:22s} {n / 1e6:7.2f} M   {"TRAIN" if t else "frozen"}')

## 3. Ajuste fino

- `lr` es la tasa de la cabeza. El stage 4 forma parte del backbone, así que usa `lr × backbone_lr_mult`.
  Con menos parámetros libres se puede usar una tasa algo mayor que en el ajuste completo: aquí cabeza 1e-3 y stage 4 1e-4.
- Lote efectivo: `batch_size × accumulation_steps` = 4 × 4 = 16 miniclips.
- Pérdida: se elige sola según la cabeza (`BCEWithLogitsLoss` con sigmoide, `CrossEntropyLoss` con softmax).
  Si las clases están desbalanceadas, con sigmoide usa `pos_weight = n_no_explicito / n_explicito`,
  y con softmax usa `class_weights`. Con `balanced_sampling=True` el muestreador ya equilibra las clases, así que déjalo en `None`.
- `best.pth` se elige con **val** (`monitor`).
- `freeze_backbone_epochs` se deja en 0. Al terminar esa fase, el paquete descongelaría **todo** el backbone,
  incluidos los stages 1–3, así que no se combina con `frozen_stages`.
- Si Colab se desconecta, vuelve a ejecutar desde la sección 1: `FineTuner` continúa desde `RUN_DIR/last.pth`.
- Usa un `RUN_DIR` distinto para cada entrenamiento (p. ej. cambiando `seed0` por `seed1`), o se reanudará el anterior.

In [ ]:
cfg = vswin.FineTuneConfig(
    output_dir=RUN_DIR,
    epochs=20,
    lr=1e-3,                 # cabeza
    backbone_lr_mult=0.1,    # stage 4 + norm final: 1e-4
    weight_decay=0.05,
    warmup_epochs=2,
    accumulation_steps=4,
    label_smoothing=0.0,
    pos_weight=None,         # solo con BINARY_HEAD=True (peso de la clase explícito)
    freeze_backbone_epochs=0,
    eval_batch_size=8,
    monitor='video_acc',     # o 'video_f1', 'video_auc', 'video_balanced_acc'
    seed=SEED,
    class_names=CLASS_NAMES,
    extra={'frozen_stages': FROZEN_STAGES, 'variant': VARIANT, 'binary_head': BINARY_HEAD,
           'split_dir': SPLIT_DIR},
)
tuner = vswin.FineTuner(model, train_loader, val_loader, cfg, device=device)
print('loss:', tuner.criterion)
history = tuner.fit()

In [ ]:
import matplotlib.pyplot as plt

ep = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, [h['train_loss'] for h in history], label='train loss')
ax[0].set_xlabel('epoch'); ax[0].legend()
for key in ['train_clip_acc', 'clip_acc', 'video_acc', 'video_f1']:
    pts = [(h['epoch'], h[key]) for h in history if key in h]
    if pts:
        ax[1].plot(*zip(*pts), marker='o', ms=3, label=key)
ax[1].set_xlabel('epoch'); ax[1].set_ylim(0, 1); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Validación del mejor modelo

La probabilidad de un video es el promedio de p(explícito) de sus K miniclips: σ(z) con la cabeza binaria, softmax con 2 salidas.
Se clasifica como explícito si ese promedio es mayor que 0.5.

In [ ]:
def report(metrics, rows, title):
    print(f'== {title}')
    for k, v in metrics.items():
        print(f'{k:22s} {v:.4f}')
    cm = vswin.confusion_matrix([r['label'] for r in rows], [r['pred'] for r in rows], 2)
    print('\nconfusion matrix (rows = true, cols = predicted):')
    print(f'{"":14s}{CLASS_NAMES[0]:>14s}{CLASS_NAMES[1]:>12s}')
    for i, name in enumerate(CLASS_NAMES):
        print(f'{name:14s}{cm[i, 0]:14d}{cm[i, 1]:12d}')
    wrong = sorted((r for r in rows if r['pred'] != r['label']), key=lambda r: -max(r['probs']))
    if wrong:
        print('\nmisclassified (most confident first):')
    for r in wrong[:10]:
        print(' ', r['video_id'], 'true', CLASS_NAMES[r['label']], 'p(explicito)=%.3f' % r['probs'][1], 'K=%d' % r['num_clips'])
    return cm

tuner.load_best()
val_metrics, val_rows = vswin.evaluate(model, val_loader, device, batch_size=8)
report(val_metrics, val_rows, 'validation (best.pth)');

## 5. Test (una sola vez, al final)

Estas son las métricas que se reportan. Se guardan en `RUN_DIR/test_results.json` junto con la predicción de cada video.
No cambies hiperparámetros a partir de este resultado: para eso está val.

In [ ]:
test_metrics, test_rows = vswin.evaluate(model, test_loader, device, batch_size=8)
cm = report(test_metrics, test_rows, 'test (best.pth)')

with open(f'{RUN_DIR}/test_results.json', 'w', encoding='utf-8') as f:
    json.dump({'split_dir': SPLIT_DIR, 'seed': SEED, 'binary_head': BINARY_HEAD,
               'frozen_stages': FROZEN_STAGES, 'variant': VARIANT,
               'val_metrics': val_metrics, 'test_metrics': test_metrics,
               'confusion_matrix': cm.tolist(), 'videos': test_rows}, f, indent=1)
print('saved', f'{RUN_DIR}/test_results.json')

## 6. Usar el modelo afinado en una sesión nueva

In [ ]:
# binary debe coincidir con el usado al entrenar (la cabeza tiene 1 o 2 salidas)
detector = vswin.build_model(VARIANT, num_classes=2, binary=BINARY_HEAD,
                             pretrained=f'{RUN_DIR}/best.pth', device=device).eval()

rec = test_rec[0]
clips = vswin.read_miniclips(rec.clip_paths)               # (K, 32, 224, 224, 3) uint8
pred = vswin.predict_clips(detector, clips, batch_size=8)
print(rec.video_id, '| true:', CLASS_NAMES[rec.label], '| predicted:', CLASS_NAMES[pred.label],
      '| p(explicito) = %.3f' % pred.probs[1])
print('p(explicito) por miniclip:', pred.clip_probs[:, 1].round(3))